In [1]:
import numpy as np

np.random.seed(42)
n = 200
X = np.random.randn(n, 2)
true_beta = np.array([
    [2.5],
    [-1.8]
])

bias = -0.5
z = X @ true_beta + bias # generating the linear response ( xbeta) 
p = 1 / (1 + np.exp(-z)) # converting that into probabilities using sigmoid

# each element becomes 1 with probabiliy of p and 0 with probability of 1-p
y = np.random.binomial(1, p).reshape(-1, 1) # converting probabilities to binomial 

print(f"X shape = {X.shape}")
print(f"y shape = {y.shape}")

X shape = (200, 2)
y shape = (200, 1)


In [2]:
X[:10]

array([[ 0.49671415, -0.1382643 ],
       [ 0.64768854,  1.52302986],
       [-0.23415337, -0.23413696],
       [ 1.57921282,  0.76743473],
       [-0.46947439,  0.54256004],
       [-0.46341769, -0.46572975],
       [ 0.24196227, -1.91328024],
       [-1.72491783, -0.56228753],
       [-1.01283112,  0.31424733],
       [-0.90802408, -1.4123037 ]])

### Logistic Regression Fitting

 * Lostic regression models the probability of a binary outcome as a smooth function of predictors, constrained between 0 and 1 .
   $$
   P(Y=y) = p^y(1-p)^{1-y}
   $$
   $$
\log\left(\frac{p}{1-p}\right) = X\beta
$$
 * Maximising log likelihood ( minimising binary cross entropy) 
$$
\text{Binary Cross Entropy} = - \left(y_i \log(p_i) + (1-y_i)\log(1-p_i) \right)
$$

In [3]:
import numpy as np
from scipy.stats import norm, chi2


class LogisticRegression:
    
    def __init__(self,lr=0.01,n_iterations=1000,fit_intercept=True,threshold=0.5):

        self.lr = lr
        self.n_iterations = n_iterations
        self.fit_intercept = fit_intercept
        self.threshold = threshold

        self.coef_ = None


    def _add_intercept(self, X):

        if self.fit_intercept:
            intercept = np.ones((X.shape[0], 1))
            return np.hstack((intercept, X))

        return X


    def _sigmoid(self, z):
        return 1 / (1 + np.exp(-z))


    def _binary_cross_entropy(self, y, y_pred):
        epsilon = 1e-15
        y_pred = np.clip(y_pred, epsilon, 1 - epsilon)

        loss = -np.mean(y * np.log(y_pred)+(1 - y) * np.log(1 - y_pred))
        return loss


    def _log_likelihood(self, y, y_pred):
        epsilon = 1e-15
        y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
        ll = np.sum(y * np.log(y_pred)+(1 - y) * np.log(1 - y_pred))
        return ll

    
    def fit(self, X, y):

        X = np.asarray(X)
        y = np.asarray(y).reshape(-1, 1)

        X = self._add_intercept(X)

        self.X = X
        self.y = y

        self.n_samples, self.n_features = X.shape
        self.coef_ = np.zeros((self.n_features, 1))

        for i in range(self.n_iterations):

            z = X @ self.coef_
            y_pred = self._sigmoid(z)
            loss = self._binary_cross_entropy(y, y_pred)
            print(f"Epoch = {i+1} ; BCE Loss = {loss:.6f}")
            grad = (1 / self.n_samples) * (X.T @ (y_pred - y))
            self.coef_ -= self.lr * grad

        self._compute_standard_errors()
        self._hypothesis_testing()
        self._likelihood_ratio_test()

        return self


    def predict_proba(self, X):
        X = self._add_intercept(X)
        z = X @ self.coef_
        return self._sigmoid(z)


    def predict(self, X):
        probabilities = self.predict_proba(X)
        return (probabilities >= self.threshold).astype(int)


    def predict_one(self, x):
        if self.fit_intercept:
            x = np.hstack(([1.0], x))

        probability = self._sigmoid(x @ self.coef_)
        return probability


    def score(self, X, y):
        y = np.asarray(y).reshape(-1, 1)
        predictions = self.predict(X)
        accuracy = np.mean(predictions == y)
        return accuracy

    def _compute_standard_errors(self):
        self.predicted_probabilities = self.predict_proba(
            self.X[:, 1:] if self.fit_intercept else self.X
        )

        p = self.predicted_probabilities

        # W matrix
        W = np.diag((p * (1 - p)).flatten())

        # covariance matrix
        self.covariance_matrix = np.linalg.pinv(self.X.T @ W @ self.X)

        # standard errors
        self.standard_errors = np.sqrt(np.diag(self.covariance_matrix)).reshape(-1, 1)


    def _hypothesis_testing(self, alpha=0.05):

        self.z_statistics = (self.coef_ / self.standard_errors)
        self.p_values = 2 * (1 - norm.cdf(np.abs(self.z_statistics)))
        critical_value = norm.ppf(1 - alpha / 2)

        self.confidence_intervals = np.hstack([
            self.coef_ - critical_value * self.standard_errors,
            self.coef_ + critical_value * self.standard_errors
        ])


    def _likelihood_ratio_test(self):

        # full model log likelihood
        full_probs = self.predict_proba(
            self.X[:, 1:] if self.fit_intercept else self.X
        )

        ll_full = self._log_likelihood(self.y, full_probs)

        # null model (intercept only)
        mean_prob = np.mean(self.y)

        null_probs = np.full_like(self.y, mean_prob)

        ll_null = self._log_likelihood(self.y, null_probs)

        # LR statistic
        self.lr_statistic = -2 * (ll_null - ll_full)

        # degrees of freedom
        self.lr_df = self.n_features - 1 if self.fit_intercept else self.n_features

        # p-value
        self.lr_p_value = 1 - chi2.cdf(
            self.lr_statistic,
            df=self.lr_df
        )

    def _mcfadden_r2(self):

        ll_full = self._log_likelihood(self.y, self.predicted_probabilities)
    
        # null model (intercept only)
        mean_prob = np.mean(self.y)
        null_probs = np.full_like(self.y, mean_prob)
    
        ll_null = self._log_likelihood(self.y, null_probs)
        self.mcfadden_r2 = 1 - (ll_full / ll_null)
    
        return self.mcfadden_r2

In [4]:
model = LogisticRegression(lr=0.01,n_iterations=1000)
model.fit(X, y)

Epoch = 1 ; BCE Loss = 0.693147
Epoch = 2 ; BCE Loss = 0.691901
Epoch = 3 ; BCE Loss = 0.690660
Epoch = 4 ; BCE Loss = 0.689425
Epoch = 5 ; BCE Loss = 0.688196
Epoch = 6 ; BCE Loss = 0.686973
Epoch = 7 ; BCE Loss = 0.685756
Epoch = 8 ; BCE Loss = 0.684545
Epoch = 9 ; BCE Loss = 0.683339
Epoch = 10 ; BCE Loss = 0.682139
Epoch = 11 ; BCE Loss = 0.680945
Epoch = 12 ; BCE Loss = 0.679756
Epoch = 13 ; BCE Loss = 0.678573
Epoch = 14 ; BCE Loss = 0.677396
Epoch = 15 ; BCE Loss = 0.676224
Epoch = 16 ; BCE Loss = 0.675058
Epoch = 17 ; BCE Loss = 0.673897
Epoch = 18 ; BCE Loss = 0.672742
Epoch = 19 ; BCE Loss = 0.671592
Epoch = 20 ; BCE Loss = 0.670448
Epoch = 21 ; BCE Loss = 0.669309
Epoch = 22 ; BCE Loss = 0.668176
Epoch = 23 ; BCE Loss = 0.667048
Epoch = 24 ; BCE Loss = 0.665925
Epoch = 25 ; BCE Loss = 0.664808
Epoch = 26 ; BCE Loss = 0.663696
Epoch = 27 ; BCE Loss = 0.662589
Epoch = 28 ; BCE Loss = 0.661488
Epoch = 29 ; BCE Loss = 0.660391
Epoch = 30 ; BCE Loss = 0.659300
Epoch = 31 ; BCE Lo

In [5]:
probability_predictions = model.predict_proba(X)
integer_predictions = model.predict(X)
accuracy = model.score(X, y)

In [6]:
model.predict_one(X[0])

array([0.63604512])

### Coefficients Interpreations
 * $\beta_1$ - change in log-odds for 1 unit increase in $x_1$
 * one unit increase in $x_1$ multiplies the odds of y = 1 by beta, holding other variables constant.

In [7]:
model.coef_

array([[-0.21035245],
       [ 1.25494339],
       [-1.05048012]])

In [8]:
np.exp(model.coef_) # odds ratios 

array([[0.8102986 ],
       [3.50763979],
       [0.34976978]])

In [9]:
(1 - np.exp(model.coef_))*100

array([[  18.97013959],
       [-250.76397918],
       [  65.02302241]])

 * One unit increase in $x_1$ reduces the odds of $y=1$ by 18%
 * One unit increase in $x_2$ multiplies the odds of $y=1$ by 3.5
 * 0.349 is the odds of $y=1$ at $x_1=x_2= 0$

**Interpretations**
 - $\beta_1$ - change in log-odds for 1 unit increase in $x_1$
 - A one -unit increase in $x_1$ multiplies the odds of $y=1$ by 16.75 , holding other variables constant. 

### Standard Errors of Logistic Regression Coefficients
 * Coefficients are estimated using **Maximum Likelihood Estimation (MLE)**.
 * Hessian / Fisher Information Matrix : $H = X^T W X$
 * Weight matrix : $W = \text{diag}(p_i(1-p_i))$
 * Covariance matrix of coefficients : $\text{Cov}(\hat{\beta}) = (X^T W X)^{-1}$
 * Standard error of coefficient $\hat{\beta}_j$ is given by : 
$$
SE(\hat{\beta}_j)=\sqrt{\left[(X^T W X)^{-1}\right]_{jj}}
$$
 * Standard errors quantify uncertainty in coefficient estimates.
 * Used in Wald Test, confidence intervals and p-value computation.

In [10]:
model.coef_

array([[-0.21035245],
       [ 1.25494339],
       [-1.05048012]])

In [11]:
 model.standard_errors

array([[0.17125261],
       [0.22550852],
       [0.21689511]])

In [12]:
model.confidence_intervals

array([[-0.54600139,  0.12529649],
       [ 0.81295481,  1.69693196],
       [-1.47558673, -0.62537352]])

### Hypothesis Testing & Likelihood Ratio Test
 * **Wald Test** : Most common coefficient significance test. $z =\frac{\hat{\beta}_j}{SE(\hat{\beta}_j)}$ . Under $H_0$: $z \sim N(0,1)$. Equivalent chi-square form:$z^2 \sim \chi^2_1$
 * **Likelihood Ratio Test** for overall model significance : $H_0:\beta_1 = \beta_2 = \cdots = \beta_p = 0$
 * In this test we compare null model (intercept only) with full model 
 * Likelihood Ratio Statistic is given by ; $LR=-2\left[\log L_0 - \log L_1\right]$
 * Under $H_0$: $LR \sim \chi^2_k$; where k is the number of predictors added.

In [13]:
model.z_statistics # from wald test

array([[-1.2283168 ],
       [ 5.56494891],
       [-4.84326328]])

In [14]:
model.p_values # corresponding p value 

array([[2.19328067e-01],
       [2.62229225e-08],
       [1.27723865e-06]])

In [15]:
model.p_values>0.05 # first coef is not significant 

array([[ True],
       [False],
       [False]])

In [16]:
model.lr_statistic

np.float64(5999.4298094089345)

In [17]:
model.lr_p_value

np.float64(0.0)

### Leverage 
 * Leverage measures how unusual an observation's predictor values are.
 * Leverage matrix : $H = W^{1/2}X(X^T W X)^{-1}X^T W^{1/2}$
 * Leverage values are diagonal elements :$h_i = H_{ii}$
 * High leverage points can strongly influence model fitting.

In [18]:
def leverage_values(X, p):
    """
    X : design matrix with intercept
    p : predicted probabilities
    """

    W = np.diag((p * (1 - p)).flatten())

    H = (
        np.sqrt(W)
        @ X
        @ np.linalg.inv(X.T @ W @ X)
        @ X.T
        @ np.sqrt(W)
    )

    leverage = np.diag(H)

    return leverage

In [19]:
leverage_values(X,model.predicted_probabilities)

array([0.00352164, 0.02060784, 0.00096699, 0.02451031, 0.00511874,
       0.00378238, 0.01815173, 0.01731043, 0.00784863, 0.02544841,
       0.01393151, 0.0158585 , 0.00336509, 0.00860811, 0.00398992,
       0.01021831, 0.01081221, 0.01265219, 0.01865669, 0.00966303,
       0.00599922, 0.00100857, 0.01800906, 0.00861136, 0.01667547,
       0.00355867, 0.00673441, 0.01738363, 0.0067675 , 0.0088987 ,
       0.00249467, 0.02305474, 0.02079586, 0.00807177, 0.0065768 ,
       0.01668264, 0.0132157 , 0.00542761, 0.00130223, 0.02016465,
       0.00219112, 0.01299953, 0.00764338, 0.00911835, 0.00530499,
       0.00787557, 0.00525079, 0.01975135, 0.00140598, 0.00065496,
       0.01369497, 0.00719389, 0.00225556, 0.01858275, 0.00102249,
       0.01204292, 0.01630103, 0.00165942, 0.01258493, 0.01681934,
       0.01137521, 0.01077979, 0.02598308, 0.01050897, 0.00327014,
       0.01097639, 0.00823594, 0.00878156, 0.0061645 , 0.01269223,
       0.01267777, 0.01052241, 0.00594941, 0.02826931, 0.00346

### Cooks distance
* Cook's Distance :
$
D_i =
\frac{
r_i^2 h_i
}{
p(1-h_i)^2
}
$
 * $r_i$ = standardized residual
 * $h_i$ = leverage
 * $p$ = number of parameters
 * Large Cook's Distance indicates influential observations.
 * Common threshold :$D_i > \frac{4}{n}$ ; ( 2,3,4 ) 


In [20]:
def cooks_distance(X, y, p):
    """
    X : design matrix
    y : true labels
    p : predicted probabilities
    """

    n, k = X.shape

    # leverage
    h = leverage_values(X, p)

    # Pearson residuals
    residuals = (y.flatten() - p.flatten()) / np.sqrt(
        p.flatten() * (1 - p.flatten())
    )

    # Cook's Distance
    D = (
        residuals**2 * h
    ) / (k * (1 - h)**2)

    return D

In [21]:
cooks_distance(X,y,model.predicted_probabilities)

array([1.01470383e-03, 3.96187861e-03, 3.74178657e-04, 4.90513662e-03,
       6.57415124e-04, 2.57915084e-03, 7.71240467e-02, 1.50490693e-03,
       6.51447675e-04, 1.17204605e-02, 1.10842252e-03, 2.07813004e-03,
       6.16932198e-04, 5.64057736e-04, 3.88188537e-03, 2.83748693e-04,
       2.28292455e-03, 7.91189575e-04, 1.17394466e-03, 6.13013014e-04,
       1.77562771e-03, 5.25496076e-04, 2.52046262e-03, 6.55942953e-04,
       1.08493082e-03, 3.26821718e-03, 6.22004990e-04, 8.10314651e-03,
       1.34160157e-03, 1.99612720e-03, 6.76615863e-04, 8.57820271e-03,
       2.00658419e-02, 1.05815074e-03, 1.32635227e-03, 2.18664824e-03,
       1.01605443e-03, 3.50093646e-05, 5.27580420e-04, 1.43164123e-03,
       4.65094098e-04, 7.47651537e-04, 7.79840279e-03, 2.56608946e-03,
       6.51674099e-04, 1.32361137e-03, 1.25680578e-03, 4.45935915e-03,
       7.89288277e-04, 3.14268286e-04, 1.50196955e-03, 4.46764115e-03,
       4.90448179e-04, 1.34082934e-03, 4.23195342e-04, 4.62602215e-04,
      

### Goodness of Fit in Logistic Regression
 * Goodness of fit measures how well the model explains observed outcomes.
 * Residual Deviance is the primary likelihood-based fit metric :
$$
D = -2(\log L_{\text{model}} - \log L_{\text{saturated}})
$$
 * Lower deviance indicates better model fit.
 * **Hosmer-Lemeshow Test** compares observed vs predicted probabilities across groups.
 * Hypotheses : $H_0 : \text{Model fits data well}$ ; $H_1 : \text{Model does not fit data well}$
   $$
HL =
\sum_{g=1}^{G}
\left(
\frac{(O_{1g}-E_{1g})^2}{E_{1g}}
+
\frac{(O_{0g}-E_{0g})^2}{E_{0g}}
\right)
$$
 * Test statistic approximately follows :$\chi^2_{g-2}$ ; where $g$ = number of groups.
 * Large p-value $(>0.05)$ indicates good fit ; small p-value $(<0.05)$ indicates poor fit.

In [34]:
np.array_split(np.random.randn(10),3)

[array([ 0.82940558, -2.21113531,  0.23561456,  0.77086519]),
 array([-1.47858625,  1.14375404,  0.33849641]),
 array([-0.41528791,  0.63278187,  2.27069286])]

In [22]:
from scipy.stats import chi2
import numpy as np

def hosmer_lemeshow_test(y_true, y_prob, g=10):
    # sort by predicted probability
    order = np.argsort(y_prob)

    y_true = np.array(y_true)[order]
    y_prob = np.array(y_prob)[order]

    # split into groups
    y_true_groups = np.array_split(y_true, g)
    y_prob_groups = np.array_split(y_prob, g)

    hl_stat = 0

    for yt, yp in zip(y_true_groups, y_prob_groups):

        observed_1 = np.sum(yt)
        expected_1 = np.sum(yp)

        observed_0 = len(yt) - observed_1
        expected_0 = len(yt) - expected_1

        hl_stat += ((observed_1 - expected_1) ** 2 / (expected_1 + 1e-10))
        hl_stat += ((observed_0 - expected_0) ** 2 / (expected_0 + 1e-10))

    # degrees of freedom = g - 2
    p_value = 1 - chi2.cdf(hl_stat, df=g - 2)

    return hl_stat, p_value

In [23]:
hosmer_lemeshow_test(y,model.predicted_probabilities,g=10) # hs statistic, p value 

(np.float64(114.44310235920398), np.float64(0.0))

### McFadden $R^2$ in Logistic Regression
 * Logistic regression has no true variance-based $R^2$.
 * McFadden $R^2$ is a likelihood-based pseudo $R^2$ measure.
$$
R^2_{\text{McFadden}}
=
1 -
\frac{
\log L_{\text{full}}
}{
\log L_{\text{null}}
}
$$
 * $\log L_{\text{full}}$ = log-likelihood of fitted model
 * $\log L_{\text{null}}$ = log-likelihood of intercept-only model
 * Larger values indicate better model fit.
 * Typical good values lie between $0.2 - 0.4$.
 * Uses the same likelihood idea as the Likelihood Ratio Test.

In [24]:
model._mcfadden_r2()

np.float64(0.9758499104090554)

### Box-Tidwell Test (Linearity of Log-Odds)

 * Used to check whether predictors have a linear relationship with log-odds in logistic regression.
 * For predictor $x$, create interaction term : $x \log(x)$
 * Extended logistic regression model :$\log\left(\frac{p}{1-p}\right)=\beta_0 + \beta_1 x + \beta_2 x\log(x)$
 * Hypotheses :
$H_0 : \beta_2 = 0$ ; $H_1 : \beta_2 \neq 0$
 * Applied separately for each continuous predictor.

In [28]:
def box_tidwell_test(X, y):
    n_features = X.shape[1]
    results = {}

    for j in range(n_features):
        x = X[:, j]
        # shift feature to make all values positive
        if np.any(x <= 0):
            x = x - np.min(x) + 1

        # interaction term
        interaction = (
            x * np.log(x)
        ).reshape(-1, 1)

        # augmented matrix
        X_aug = np.hstack([X, interaction])

        # fit model
        model = LogisticRegression()
        model.fit(X_aug, y)

        # p-value of interaction coefficient
        p_value = model.p_values[-1][0]

        results[f"x{j}"] = {
            "p_value": p_value,
            "linearity_satisfied": p_value > 0.05
        }
        print('*'*50)

    return results

In [30]:
results = box_tidwell_test(X,y)

Epoch = 1 ; BCE Loss = 0.693147
Epoch = 2 ; BCE Loss = 0.691020
Epoch = 3 ; BCE Loss = 0.689089
Epoch = 4 ; BCE Loss = 0.687314
Epoch = 5 ; BCE Loss = 0.685663
Epoch = 6 ; BCE Loss = 0.684109
Epoch = 7 ; BCE Loss = 0.682633
Epoch = 8 ; BCE Loss = 0.681220
Epoch = 9 ; BCE Loss = 0.679858
Epoch = 10 ; BCE Loss = 0.678536
Epoch = 11 ; BCE Loss = 0.677248
Epoch = 12 ; BCE Loss = 0.675987
Epoch = 13 ; BCE Loss = 0.674749
Epoch = 14 ; BCE Loss = 0.673529
Epoch = 15 ; BCE Loss = 0.672326
Epoch = 16 ; BCE Loss = 0.671136
Epoch = 17 ; BCE Loss = 0.669959
Epoch = 18 ; BCE Loss = 0.668792
Epoch = 19 ; BCE Loss = 0.667635
Epoch = 20 ; BCE Loss = 0.666486
Epoch = 21 ; BCE Loss = 0.665345
Epoch = 22 ; BCE Loss = 0.664211
Epoch = 23 ; BCE Loss = 0.663085
Epoch = 24 ; BCE Loss = 0.661964
Epoch = 25 ; BCE Loss = 0.660850
Epoch = 26 ; BCE Loss = 0.659743
Epoch = 27 ; BCE Loss = 0.658640
Epoch = 28 ; BCE Loss = 0.657544
Epoch = 29 ; BCE Loss = 0.656453
Epoch = 30 ; BCE Loss = 0.655367
Epoch = 31 ; BCE Lo

In [31]:
results

{'x0': {'p_value': np.float64(0.9715342810141938),
  'linearity_satisfied': np.True_},
 'x1': {'p_value': np.float64(0.8488184439252033),
  'linearity_satisfied': np.True_}}